![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de popularidad en canción

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de popularidad en canción".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la competencia de Kaggle cuyo link estará disponible en la sección del Coursera del proyecto.

## Datos para la predicción de popularidad en cancion

En este proyecto se usará el conjunto de datos de datos de popularidad en canciones, donde cada observación representa una canción y se tienen variables como: duración de la canción, acusticidad y tempo, entre otras. El objetivo es predecir qué tan popular es la canción. Para más detalles puede visitar el siguiente enlace: [datos](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv', index_col=0)

print(f"✓ Dataset de entrenamiento cargado: {dataTraining.shape}")
print(f"✓ Dataset de test cargado: {dataTesting.shape}")

✓ Dataset de entrenamiento cargado: (79800, 21)
✓ Dataset de test cargado: (34200, 19)


## Reescritura reproducible del pipeline

La siguiente seccion mantiene un flujo reproducible con `sklearn.Pipeline`. El flujo separa primero los datos crudos en entrenamiento y prueba holdout, de modo que toda la logica de preprocesamiento aprenda unicamente del split de entrenamiento. Se conservan las variables de ingenieria basadas en artistas, albumes, genero y banderas de texto. Para recuperar el mejor holdout reciente, esta version desactiva las dos columnas de interaccion `primary_artist_genre_popularity_score` y `album_genre_popularity_score`, y usa `artist_alpha=2`, `genre_alpha=2` y `album_alpha=10`. Las metricas finales mostradas se calculan exclusivamente sobre el split de prueba holdout.

In [112]:
dataTraining

,Unnamed: 0,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,0,7hUhmkALyQ8SX9mJs5XI3D,Love and Rockets,Love and Rockets,Motorcycle,211533,False,0.305,0.8490,9,...,1,0.0549,0.000058,0.056700,0.4640,0.3200,141.793,4,goth,22
1,1,5x59U89ZnjZXuNAAlc8X1u,Filippa Giordano,Filippa Giordano,"Addio del passato - From ""La traviata""",196000,False,0.287,0.1900,7,...,0,0.0370,0.930000,0.000356,0.0834,0.1330,83.685,4,opera,22
2,2,70Vng5jLzoJLmeLu3ayBQq,Susumu Yokota,Symbol,Purple Rose Minuet,216506,False,0.583,0.5090,1,...,1,0.0362,0.777000,0.202000,0.1150,0.5440,90.459,3,idm,37
3,3,1cRfzLJapgtwJ61xszs37b,Franz Liszt;YUNDI,Relajación y siestas,"Liebeslied (Widmung), S. 566",218346,False,0.163,0.0368,8,...,1,0.0472,0.991000,0.899000,0.1070,0.0387,69.442,3,classical,0
4,4,47d5lYjbiMy0EdMRV8lRou,Scooter,Scooter Forever,The Darkside,173160,False,0.647,0.9210,2,...,1,0.1850,0.000939,0.371000,0.1310,0.1710,137.981,4,techno,27


In [15]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

train_url = 'https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv'
dataTraining = pd.read_csv(train_url)

columnas_numericas = [
    'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
    'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'time_signature'
 ]

BEST_ALPHA_CONFIG = {
    'artist_alpha': 2,
    'genre_alpha': 2,
    'album_alpha': 10,
    'primary_artist_genre_alpha': 2,
    'album_genre_alpha': 20,
}
FINAL_RF_ESTIMATORS = 250

TRACK_FLAG_PATTERNS = {
    'track_name_has_dash': r'\s[-–]\s',
    'track_name_has_bracketed_context': r'[\(\)\[\]"“”]',
    'track_name_has_numbers': r'\b\d+(?:st|nd|rd|th)?\b',
    'track_name_has_feat': r'\b(?:feat\.?|ft\.?|featuring)\b',
    'track_name_has_live': r'\b(?:live|en vivo|ao vivo)\b',
    'track_name_has_from': r'\bfrom\b',
    'track_name_has_mix_or_remix': r'\b(?:mix|remix|rmx)\b',
    'track_name_has_version': r'\bversion\b',
    'track_name_has_remastered': r'\bremaster(?:ed)?\b',
    'track_name_has_original': r'\boriginal\b',
    'track_name_has_christmas': r'\b(?:christmas|navidad|xmas)\b',
    'track_name_has_acoustic': r'\b(?:acoustic|acustic|acustico|acústico|acustica|acústica)\b',
}

ALBUM_FLAG_PATTERNS = {
    'album_name_has_bracketed_context': r'[\(\)\[\]"“”]',
    'album_name_has_numbers': r'\b\d+(?:st|nd|rd|th)?\b',
    'album_name_has_live': r'\b(?:live|en vivo|ao vivo)\b',
    'album_name_has_volume': r'\b(?:vol\.?|volume|volumen)\s*\d*\b',
    'album_name_has_christmas': r'\b(?:christmas|navidad|xmas)\b',
    'album_name_has_soundtrack': r'\b(?:soundtrack|ost|bso)\b|original motion picture|motion picture soundtrack',
    'album_name_has_edition': r'\b(?:edition|edicion|edición|édition)\b',
    'album_name_has_deluxe': r'\bdeluxe\b',
    'album_name_has_remaster': r'\bremaster(?:ed)?\b',
    'album_name_has_version': r'\bversion\b',
    'album_name_has_acoustic': r'\b(?:acoustic|acustic|acustico|acústico|acustica|acústica)\b',
    'album_name_has_ep': r'(?<![a-z])ep(?![a-z])',
    'album_name_has_anniversary': r'\b(?:anniversary|aniversario)\b',
}

class SpotifyFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        numeric_columns,
        artist_popularity_smoothing=2,
        genre_popularity_smoothing=2,
        album_popularity_smoothing=10,
        primary_artist_genre_smoothing=2,
        album_genre_smoothing=20,
    ):
        self.numeric_columns = numeric_columns
        self.artist_popularity_smoothing = artist_popularity_smoothing
        self.genre_popularity_smoothing = genre_popularity_smoothing
        self.album_popularity_smoothing = album_popularity_smoothing
        self.primary_artist_genre_smoothing = primary_artist_genre_smoothing
        self.album_genre_smoothing = album_genre_smoothing

    @staticmethod
    def _parse_artists(artist_value):
        if pd.isna(artist_value):
            return []
        normalized_text = str(artist_value).replace(';', ',')
        artists = [artist.strip() for artist in normalized_text.split(',') if artist.strip()]
        return artists

    @staticmethod
    def _primary_artist(artist_list):
        return artist_list[0] if artist_list else 'missing'

    @staticmethod
    def _safe_text(value):
        if pd.isna(value):
            return ''
        return str(value).strip().lower()

    @staticmethod
    def _album_key(value):
        return 'missing' if pd.isna(value) else str(value).strip() or 'missing'

    @staticmethod
    def _artist_genre_key(primary_artist, genre):
        normalized_genre = 'missing' if pd.isna(genre) else genre
        return (primary_artist, normalized_genre)

    @staticmethod
    def _album_genre_key(album, genre):
        normalized_genre = 'missing' if pd.isna(genre) else genre
        return (album, normalized_genre)

    def fit(self, X, y=None):
        artist_lists = X['artists'].apply(self._parse_artists)
        primary_artist_values = artist_lists.apply(self._primary_artist)
        genre_values = X['track_genre'].fillna('missing')
        album_values = X['album_name'].apply(self._album_key)

        popularity_sum_by_artist = {}
        support_count_by_artist = {}
        popularity_sum_by_genre = {}
        popularity_count_by_genre = {}
        popularity_sum_by_album = {}
        popularity_count_by_album = {}
        popularity_sum_by_primary_artist_genre = {}
        popularity_count_by_primary_artist_genre = {}
        popularity_sum_by_album_genre = {}
        popularity_count_by_album_genre = {}
        self.global_popularity_ = float(pd.Series(y).mean()) if y is not None else 0.0

        if y is not None:
            y_series = pd.Series(y, index=X.index)
        else:
            y_series = None

        for index, artist_list in artist_lists.items():
            current_target = float(y_series.loc[index]) if y_series is not None else None
            for artist in artist_list:
                support_count_by_artist[artist] = support_count_by_artist.get(artist, 0) + 1
                if current_target is not None:
                    popularity_sum_by_artist[artist] = popularity_sum_by_artist.get(artist, 0.0) + current_target

        if y_series is not None:
            for index in X.index:
                current_target = float(y_series.loc[index])
                genre = genre_values.loc[index]
                album = album_values.loc[index]
                primary_artist = primary_artist_values.loc[index]

                popularity_sum_by_genre[genre] = popularity_sum_by_genre.get(genre, 0.0) + current_target
                popularity_count_by_genre[genre] = popularity_count_by_genre.get(genre, 0) + 1

                popularity_sum_by_album[album] = popularity_sum_by_album.get(album, 0.0) + current_target
                popularity_count_by_album[album] = popularity_count_by_album.get(album, 0) + 1

                artist_genre_key = self._artist_genre_key(primary_artist, genre)
                popularity_sum_by_primary_artist_genre[artist_genre_key] = (
                    popularity_sum_by_primary_artist_genre.get(artist_genre_key, 0.0) + current_target
                )
                popularity_count_by_primary_artist_genre[artist_genre_key] = (
                    popularity_count_by_primary_artist_genre.get(artist_genre_key, 0) + 1
                )

                album_genre_key = self._album_genre_key(album, genre)
                popularity_sum_by_album_genre[album_genre_key] = (
                    popularity_sum_by_album_genre.get(album_genre_key, 0.0) + current_target
                )
                popularity_count_by_album_genre[album_genre_key] = (
                    popularity_count_by_album_genre.get(album_genre_key, 0) + 1
                )

        self.artist_support_dict_ = {
            artist: float(artist_count)
            for artist, artist_count in support_count_by_artist.items()
        }
        self.artist_popularity_dict_ = {}
        for artist, artist_count in support_count_by_artist.items():
            artist_mean = popularity_sum_by_artist[artist] / artist_count
            smoothed_score = (
                artist_count * artist_mean + self.artist_popularity_smoothing * self.global_popularity_
            ) / (artist_count + self.artist_popularity_smoothing)
            self.artist_popularity_dict_[artist] = float(smoothed_score)

        self.genre_popularity_dict_ = {}
        for genre, genre_count in popularity_count_by_genre.items():
            genre_mean = popularity_sum_by_genre[genre] / genre_count
            smoothed_score = (
                genre_count * genre_mean + self.genre_popularity_smoothing * self.global_popularity_
            ) / (genre_count + self.genre_popularity_smoothing)
            self.genre_popularity_dict_[genre] = float(smoothed_score)

        self.album_popularity_dict_ = {}
        for album, album_count in popularity_count_by_album.items():
            album_mean = popularity_sum_by_album[album] / album_count
            smoothed_score = (
                album_count * album_mean + self.album_popularity_smoothing * self.global_popularity_
            ) / (album_count + self.album_popularity_smoothing)
            self.album_popularity_dict_[album] = float(smoothed_score)

        self.primary_artist_genre_popularity_dict_ = {}
        for interaction_key, interaction_count in popularity_count_by_primary_artist_genre.items():
            primary_artist, genre = interaction_key
            interaction_mean = popularity_sum_by_primary_artist_genre[interaction_key] / interaction_count
            hierarchical_prior = (self._artist_score(primary_artist) + self._genre_score(genre)) / 2
            smoothed_score = (
                interaction_count * interaction_mean + self.primary_artist_genre_smoothing * hierarchical_prior
            ) / (interaction_count + self.primary_artist_genre_smoothing)
            self.primary_artist_genre_popularity_dict_[interaction_key] = float(smoothed_score)

        self.album_genre_popularity_dict_ = {}
        for interaction_key, interaction_count in popularity_count_by_album_genre.items():
            album, genre = interaction_key
            interaction_mean = popularity_sum_by_album_genre[interaction_key] / interaction_count
            hierarchical_prior = (self._album_score(album) + self._genre_score(genre)) / 2
            smoothed_score = (
                interaction_count * interaction_mean + self.album_genre_smoothing * hierarchical_prior
            ) / (interaction_count + self.album_genre_smoothing)
            self.album_genre_popularity_dict_[interaction_key] = float(smoothed_score)

        return self

    def _artist_scores(self, artist_list):
        return [self._artist_score(artist) for artist in artist_list]

    def _artist_supports(self, artist_list):
        return [self._artist_support(artist) for artist in artist_list]

    def _artist_score(self, artist):
        return float(self.artist_popularity_dict_.get(artist, self.global_popularity_))

    def _artist_support(self, artist):
        return float(self.artist_support_dict_.get(artist, 0))

    def _genre_score(self, genre):
        normalized_genre = 'missing' if pd.isna(genre) else genre
        return float(self.genre_popularity_dict_.get(normalized_genre, self.global_popularity_))

    def _primary_artist_genre_score(self, primary_artist, genre):
        interaction_key = self._artist_genre_key(primary_artist, genre)
        hierarchical_prior = (self._artist_score(primary_artist) + self._genre_score(genre)) / 2
        return float(self.primary_artist_genre_popularity_dict_.get(interaction_key, hierarchical_prior))

    def _album_genre_score(self, album, genre):
        normalized_album = self._album_key(album)
        interaction_key = self._album_genre_key(normalized_album, genre)
        hierarchical_prior = (self._album_score(normalized_album) + self._genre_score(genre)) / 2
        return float(self.album_genre_popularity_dict_.get(interaction_key, hierarchical_prior))

    def _album_score(self, album):
        normalized_album = self._album_key(album)
        return float(self.album_popularity_dict_.get(normalized_album, self.global_popularity_))

    def _artist_popularity_with_support(self, artist_list):
        if not artist_list:
            return self.global_popularity_, 0.0

        best_artist = None
        best_score = float('-inf')
        for artist in artist_list:
            current_score = self._artist_score(artist)
            if current_score > best_score:
                best_score = current_score
                best_artist = artist

        best_support = self._artist_support(best_artist)
        return float(best_score), float(best_support)

    def _primary_artist_with_support(self, artist_list):
        if not artist_list:
            return self.global_popularity_, 0.0
        primary_artist = artist_list[0]
        return self._artist_score(primary_artist), self._artist_support(primary_artist)

    def _ordered_artist_aggregates(self, artist_list):
        if not artist_list:
            return self.global_popularity_, 0.0

        scores = np.array(self._artist_scores(artist_list), dtype=float)
        supports = np.array(self._artist_supports(artist_list), dtype=float)
        weights = np.array([1.0 / (index + 1) for index in range(len(artist_list))], dtype=float)

        ordered_score = float(np.average(scores, weights=weights))
        ordered_support = float(np.average(supports, weights=weights))
        return ordered_score, ordered_support

    def _artist_gap_primary_vs_best(self, artist_list):
        if not artist_list:
            return 0.0
        primary_score = self._artist_score(artist_list[0])
        best_score = max(self._artist_scores(artist_list))
        return float(primary_score - best_score)

    def transform(self, X):
        transformed = pd.DataFrame(index=X.index)

        numeric_frame = X[self.numeric_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
        transformed[self.numeric_columns] = numeric_frame

        artist_lists = X['artists'].apply(self._parse_artists)
        primary_artist_values = artist_lists.apply(self._primary_artist)
        genre_values = X['track_genre'].fillna('missing')
        album_values = X['album_name'].apply(self._album_key)

        artist_score_support = artist_lists.apply(self._artist_popularity_with_support)
        transformed['artist_popularity_score'] = artist_score_support.apply(lambda value: value[0])
        transformed['artist_support_for_max'] = artist_score_support.apply(lambda value: value[1])

        primary_artist_features = artist_lists.apply(self._primary_artist_with_support)
        transformed['artist_popularity_primary'] = primary_artist_features.apply(lambda value: value[0])
        transformed['primary_artist_genre_popularity_score'] = [
            self._primary_artist_genre_score(primary_artist, genre)
            for primary_artist, genre in zip(primary_artist_values, genre_values)
        ]
        transformed['album_genre_popularity_score'] = [
            self._album_genre_score(album, genre)
            for album, genre in zip(album_values, genre_values)
        ]

        ordered_artist_features = artist_lists.apply(self._ordered_artist_aggregates)
        transformed['artist_popularity_ordered_mean'] = ordered_artist_features.apply(lambda value: value[0])
        transformed['artist_support_ordered_mean'] = ordered_artist_features.apply(lambda value: value[1])

        transformed['artist_gap_primary_vs_best'] = artist_lists.apply(self._artist_gap_primary_vs_best)

        track_names = X['track_name'].apply(self._safe_text)
        for feature_name, pattern in TRACK_FLAG_PATTERNS.items():
            transformed[feature_name] = track_names.str.contains(pattern, regex=True).astype(float)

        album_names = X['album_name'].apply(self._safe_text)
        for feature_name, pattern in ALBUM_FLAG_PATTERNS.items():
            transformed[feature_name] = album_names.str.contains(pattern, regex=True).astype(float)

        transformed['genre_popularity_score'] = genre_values.apply(self._genre_score)
        transformed['album_popularity_score'] = album_values.apply(self._album_score)
        transformed['primary_artist_genre_popularity_delta_vs_genre'] = (
            transformed['primary_artist_genre_popularity_score'] - transformed['genre_popularity_score']
        )
        transformed['album_genre_popularity_delta_vs_album'] = (
            transformed['album_genre_popularity_score'] - transformed['album_popularity_score']
        )

        return transformed

engineered_numeric_columns = columnas_numericas + [
    'artist_popularity_score',
    'artist_support_for_max',
    'artist_popularity_primary',
    'primary_artist_genre_popularity_score',
    'album_genre_popularity_score',
    'primary_artist_genre_popularity_delta_vs_genre',
    'album_genre_popularity_delta_vs_album',
    'artist_popularity_ordered_mean',
    'artist_support_ordered_mean',
    'artist_gap_primary_vs_best',
    'track_name_has_dash',
    'track_name_has_bracketed_context',
    'track_name_has_numbers',
    'track_name_has_feat',
    'track_name_has_live',
    'track_name_has_from',
    'track_name_has_mix_or_remix',
    'track_name_has_version',
    'track_name_has_remastered',
    'track_name_has_original',
    'track_name_has_christmas',
    'track_name_has_acoustic',
    'album_name_has_bracketed_context',
    'album_name_has_numbers',
    'album_name_has_live',
    'album_name_has_volume',
    'album_name_has_christmas',
    'album_name_has_soundtrack',
    'album_name_has_edition',
    'album_name_has_deluxe',
    'album_name_has_remaster',
    'album_name_has_version',
    'album_name_has_acoustic',
    'album_name_has_ep',
    'album_name_has_anniversary',
    'genre_popularity_score',
    'album_popularity_score',
]

def build_random_forest_pipeline(alpha_config=None, n_estimators=FINAL_RF_ESTIMATORS):
    if alpha_config is None:
        alpha_config = BEST_ALPHA_CONFIG

    preprocessor = ColumnTransformer(
        transformers=[('numeric', 'passthrough', engineered_numeric_columns)],
        remainder='drop'
    )

    return Pipeline(
        steps=[
            ('feature_builder', SpotifyFeatureBuilder(
                columnas_numericas,
                artist_popularity_smoothing=alpha_config['artist_alpha'],
                genre_popularity_smoothing=alpha_config['genre_alpha'],
                album_popularity_smoothing=alpha_config['album_alpha'],
                primary_artist_genre_smoothing=alpha_config['primary_artist_genre_alpha'],
                album_genre_smoothing=alpha_config['album_genre_alpha'],
            )),
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(
                n_estimators=n_estimators,
                n_jobs=-1,
                random_state=42,
                max_features='log2',
            )),
        ]
    )

X = dataTraining.drop(columns=['popularity'])
y = dataTraining['popularity']

X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_rf_final = build_random_forest_pipeline()
model_rf_holdout = clone(model_rf_final)
model_rf_holdout.fit(X_train_split, y_train_split)
y_pred_test_split = model_rf_holdout.predict(X_test_split)

rmse_test_rf = np.sqrt(mean_squared_error(y_test_split, y_pred_test_split))
r2_test_rf = r2_score(y_test_split, y_pred_test_split)

model_rf_final.fit(X, y)

print('Configuracion alpha fijada:', BEST_ALPHA_CONFIG)
print(f'RMSE test holdout: {rmse_test_rf:.4f}')
print(f'R² test holdout: {r2_test_rf:.4f}')
print(f'Filas train split: {X_train_split.shape[0]}')
print(f'Filas test split: {X_test_split.shape[0]}')
print(f'Variables crudas de entrada: {X.shape[1]}')

Configuracion alpha fijada: {'artist_alpha': 2, 'genre_alpha': 2, 'album_alpha': 10, 'primary_artist_genre_alpha': 2, 'album_genre_alpha': 20}
RMSE test holdout: 9.8857
R² test holdout: 0.8023
Filas train split: 63840
Filas test split: 15960
Variables crudas de entrada: 20


In [16]:
test_url = 'https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv'
dataTesting = pd.read_csv(test_url, index_col=0)

y_pred_test = model_rf_final.predict(dataTesting)
submissions = pd.DataFrame({'ID': dataTesting.index, 'Popularity': y_pred_test})

print('\nPredicciones generadas con Random Forest reproducible')
print(f'Forma: {y_pred_test.shape}')
print(f'Media popularidad predicha: {y_pred_test.mean():.2f}')
print(f'Rango: {y_pred_test.min():.2f} - {y_pred_test.max():.2f}')
submissions.head()


Predicciones generadas con Random Forest reproducible
Forma: (34200,)
Media popularidad predicha: 32.16
Rango: 0.00 - 95.72


,ID,Popularity
0,0,49.076
1,1,13.976
2,2,0.768
3,3,0.200
4,4,31.568


In [17]:
submissions = pd.DataFrame({'ID': dataTesting.index,'Popularity': y_pred_test})
submissions.to_csv('test_submission_Spotify_predictions.csv', index=False)